#Baseline - NER на Spacy

In [1]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=9cb34daa874999bd62b1af77b070630b1454032a5347212839daab4fa54ef51c
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
import json
from pathlib import Path
from typing import List, Tuple
import statistics as S
from datasets import load_from_disk
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

import spacy
from spacy.tokens import Doc
from spacy.pipeline import EntityRuler

Словари для выделения сущностей

In [14]:
ITEMS = ["платье","юбка","брюки","джинсы","топ","рубашка","пиджак","жакет",
      "сумка","очки","туфли","ботинки","шорты","блузка","купальник"]

FEATS = ["черный","черная","черные","черное","белый","белая","белые","синий","синяя",
    "голубой","красный","красная","бордовый","бежевый","беж","хаки","серый","серая"]

EVENT_PHRASES = ["офис", "свидание", "вечеринку", "пляж", "выпускной"]

In [15]:
def build_nlp():
    nlp = spacy.blank("ru")
    ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True})

    patterns = []
    patterns += [{"label":"ITEM","pattern":[{"LOWER": w}]} for w in ITEMS]
    patterns += [{"label":"FEAT","pattern":[{"LOWER": w}]} for w in FEATS]
    for a in EVENT_PHRASES:
        patterns.append({"label":"EVENT","pattern":[{"LOWER": a}]})

    ruler.add_patterns(patterns)
    return nlp

def ents_to_bio(doc: Doc) -> List[str]:
    tags = ["O"] * len(doc)
    for ent in doc.ents:
        tags[ent.start] = f"B-{ent.label_}"
        for i in range(ent.start + 1, ent.end):
            tags[i] = f"I-{ent.label_}"
    return tags

def ents_from_spacy(doc: Doc) -> List[Tuple[str,int,int]]:
    return [(ent.label_, ent.start, ent.end) for ent in doc.ents]

def strict_prf(real, pred):
    real = set(real); pred = set(pred)
    tp = len(real & pred)
    fp = len(pred - real)
    fn = len(real - pred)
    p = tp/(tp+fp) if tp+fp else 0.0
    r = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f1

In [16]:
def ids_to_tags(example, label_names):
    ids = example["tags"]
    return {"ner_tags": [label_names[i] for i in ids]}

In [17]:
ds_dir = '/content/drive/MyDrive/Colab Notebooks/dl/proj_fashion_ner/fashion_dataset_dict'
show_samples = 3

ds = load_from_disk(ds_dir)
test_ds = ds["test"]

label_names = test_ds.features["tags"].feature.names

test_ds = test_ds.map(lambda e: ids_to_tags(e, label_names))

nlp = build_nlp()

y_true, y_pred = [], []
previews = 0

for ex in test_ds:
    tokens = ex["tokens"]
    gold_tags = ex["ner_tags"]

    doc = Doc(nlp.vocab, words=tokens)
    doc = nlp(doc)
    pred_tags = ents_to_bio(doc)

    y_true.append(gold_tags)
    y_pred.append(pred_tags)

    if previews < show_samples:
        print("TOK :", tokens)
        print("GOLD:", gold_tags)
        print("PRED:", pred_tags)
        print("-"*70)
        previews += 1

# Метрики entity-level от seqeval
p = precision_score(y_true, y_pred)
r = recall_score(y_true, y_pred)
f = f1_score(y_true, y_pred)
print(f"\nSeqeval entity-level (micro): P={p:.3f} R={r:.3f} F1={f:.3f}\n")
print(classification_report(y_true, y_pred, digits=3))



TOK : ['А', 'синий', 'колготки', 'где', 'покупали', ',', 'киньте', 'ссыль']
GOLD: ['O', 'B-FEAT', 'B-ITEM', 'O', 'O', 'O', 'O', 'O']
PRED: ['O', 'B-FEAT', 'O', 'O', 'O', 'O', 'O', 'O']
----------------------------------------------------------------------
TOK : ['Ищу', 'черное', 'ремень', 'как', 'у', 'вас', ',', 'можно', 'ссылку']
GOLD: ['O', 'B-FEAT', 'B-ITEM', 'O', 'O', 'O', 'O', 'O', 'O']
PRED: ['O', 'B-FEAT', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
----------------------------------------------------------------------
TOK : ['Какие', 'параметры', 'у', 'темно-синяя', 'джинсы', '?', 'талия', 'и', 'бедра', 'интересуют']
GOLD: ['O', 'O', 'O', 'B-FEAT', 'B-ITEM', 'O', 'B-FEAT', 'O', 'B-FEAT', 'O']
PRED: ['O', 'O', 'O', 'O', 'B-ITEM', 'O', 'O', 'O', 'O', 'O']
----------------------------------------------------------------------

Seqeval entity-level (micro): P=0.988 R=0.197 F1=0.328

              precision    recall  f1-score   support

       EVENT      1.000     0.135     0.237        52
